# R3 Kaggle runner: one run step (K0 to K6) of the Gemma Scope ladder

Pre-registration: `results/gemma2_pile/r3_gemma_scope_ladder/README.md` (Section 1, **Amendment 1**, Clarification 1).
This notebook runs **one** step of `scripts/gemma/05_pile_gemma_scope_ladder.py` per Kaggle session, then the
**V10 secret scan**. The CPU steps (`--aggregate`, script 06, script 07) are run later, outside Kaggle.

**Before each run (notebook settings):**
1. Accelerator: **GPU T4 x2**. Internet: **on** (needs a phone-verified Kaggle account).
2. Add-ons → Secrets: a secret named **`HF_TOKEN`** (a *read* token of the Hugging Face account that accepted the
   Gemma licence), attached to this notebook only. The token is never typed into a cell.
3. Keep this notebook **private**.
4. Fill in the settings cell: `RUN`, `COMMIT`, and for K1 to K6 also `WEEK_USED_HOURS` (from the Kaggle quota display) and
   `RESUME_FROM` (the previous run's output, added with "Add Input").
5. Save Version → **Save & Run All (Commit)**, so the run continues in the background for up to 12 h.

**After the run:** check the last cell's status. Outputs are in `/kaggle/working/r3_out/`. Download them only if
V10 passed. If V10 failed: do not download or commit anything, revoke the token on huggingface.co and make a new one.

Commit this notebook only with all outputs cleared.

In [ ]:
# ---- settings (the only cell to edit) -------------------------------------------------------------
RUN = 'K0'                 # one of K0, K1, K2, K3, K4, K5, K6 (README, Amendment 1 (4))
COMMIT = 'SET_ME'          # full 40-character commit of branch simba-gemma-m6-results that holds the R3 code
WEEK_USED_HOURS = None     # K1-K6: GPU hours already used this week (Kaggle quota display), e.g. 1.2
RESUME_FROM = None         # K1-K6: previous run's output root, e.g. '/kaggle/input/<previous-version>/r3_out'
WIDTHS = None              # optional: split a run by width, e.g. ['16k', '32k'] (recorded by script 05)

REPO_URL = 'https://github.com/Fall2026group4/fall-2026-group4'   # public; no GitHub token needed
BRANCH = 'simba-gemma-m6-results'
REPO_DIR = '/tmp/r3_repo'           # code lives outside /kaggle/working, so it is not saved as output
OUT = '/kaggle/working/r3_out'      # everything Kaggle should keep goes here
HF_CACHE = '/tmp/hf_cache'          # model and SAE downloads: not output (script 05 deletes each SAE after use)

import re
assert RUN in ('K0', 'K1', 'K2', 'K3', 'K4', 'K5', 'K6'), RUN
assert re.fullmatch(r'[0-9a-f]{40}', COMMIT), 'set COMMIT to the full commit hash of the R3 code'
if RUN != 'K0':
    assert WEEK_USED_HOURS is not None, 'K1-K6 need WEEK_USED_HOURS (budget rule)'
print('settings ok:', RUN, COMMIT[:12])

In [ ]:
# ---- code at the pinned commit -------------------------------------------------------------------
import os, subprocess
subprocess.run(['rm', '-rf', REPO_DIR], check=True)
subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, '-c', 'advice.detachedHead=false', 'checkout', '--quiet', COMMIT], check=True)
head = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True,
                      check=True).stdout.strip()
dirty = subprocess.run(['git', '-C', REPO_DIR, 'status', '--porcelain'], capture_output=True, text=True,
                       check=True).stdout.strip()
assert head == COMMIT and not dirty, (head, dirty)
print('code at', head)

In [ ]:
# ---- pinned software (same torch and transformers as M6; README Amendment 1) ----------------------
# Nothing has imported torch in this kernel yet, so the pinned version is the one that gets loaded.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'torch==2.7.0',
                '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers==5.21.0', 'zstandard'], check=True)
print(subprocess.run([sys.executable, '-c', 'import torch, transformers; print("torch", torch.__version__, '
                      '"| transformers", transformers.__version__, "| CUDA", torch.version.cuda, "| GPUs", '
                      '[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])'],
                     capture_output=True, text=True, check=True).stdout)
# If a version differs from torch 2.7.0+cu128 / transformers 5.21.0, script 05 records it (V0) as a deviation.

In [ ]:
# ---- Hugging Face token from Kaggle Secrets (README Amendment 1 (6)) ------------------------------
# Read into this process's memory only. Never printed, never written to a file, never passed on a command line.
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_HOME'] = HF_CACHE
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # before torch is first imported (determinism)
print('HF_TOKEN loaded from Kaggle Secrets:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
# ---- run one step of script 05, with a copy of everything printed kept in OUT/logs ----------------
import importlib.util, io, os, sys, traceback
from datetime import datetime, timezone
from pathlib import Path

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
Path(OUT, 'logs').mkdir(parents=True, exist_ok=True)
log_path = Path(OUT, 'logs', f'{RUN}_{stamp}.log')


class Tee(io.TextIOBase):
    def __init__(self, *streams):
        self.streams = streams

    def write(self, s):
        for st in self.streams:
            st.write(s)
            st.flush()
        return len(s)


argv = ['--run', RUN, '--out', OUT]
if RESUME_FROM:
    argv += ['--resume-from', RESUME_FROM]
if WEEK_USED_HOURS is not None:
    argv += ['--week-used-hours', str(WEEK_USED_HOURS)]
if WIDTHS:
    argv += ['--widths', *WIDTHS]

run_status = 'not started'
os.chdir(REPO_DIR)
with open(log_path, 'w', encoding='utf-8') as log:
    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = Tee(old_out, log), Tee(old_err, log)
    try:
        print('script 05 arguments:', argv)
        spec = importlib.util.spec_from_file_location('s05', f'{REPO_DIR}/scripts/gemma/05_pile_gemma_scope_ladder.py')
        s05 = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(s05)
        s05.main(argv)
        run_status = 'finished'
    except SystemExit as exc:   # a failed gate or a refused start: keep going so V10 still runs
        run_status = f'stopped: {exc}'
    except Exception:           # any crash: same, with the traceback in the log
        traceback.print_exc()
        run_status = 'crashed (see log)'
    finally:
        sys.stdout, sys.stderr = old_out, old_err
print('run status:', run_status)
print('log:', log_path)

In [ ]:
# ---- V10: no secret in anything Kaggle will keep (README Amendment 1 (6)) --------------------------
# Searches every file under /kaggle/working (all outputs, including the log above) for the exact token and for
# anything shaped like a Hugging Face token. Only file names and the kind of hit are printed, never the text.
import json, os
from datetime import datetime, timezone
from pathlib import Path

if 's05' not in globals():  # the run cell failed before loading script 05
    spec = importlib.util.spec_from_file_location('s05', f'{REPO_DIR}/scripts/gemma/05_pile_gemma_scope_ladder.py')
    s05 = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(s05)
hits = s05.secret_scan(Path('/kaggle/working'), os.environ.get('HF_TOKEN'))
v10 = {'gate': 'V10', 'pass': not hits, 'hits': hits, 'scanned': '/kaggle/working', 'run': RUN,
       'run_status': run_status, 'checked_utc': datetime.now(timezone.utc).isoformat(timespec='seconds')}
Path(OUT, 'v10').mkdir(parents=True, exist_ok=True)
Path(OUT, 'v10', f'{RUN}_{stamp}.json').write_text(json.dumps(v10, indent=2) + '\n', encoding='utf-8')
if hits:
    Path(OUT, 'V10_FAILED.json').write_text(json.dumps(v10, indent=2) + '\n', encoding='utf-8')
os.environ.pop('HF_TOKEN', None)   # the token is not needed after this point

print('V10 secret scan:', 'PASS' if not hits else 'FAILED')
for h in hits:
    print('  hit:', h['kind'], 'in', h['file'])
if hits:
    print('Do NOT download or commit these outputs. Revoke the token on huggingface.co and create a new one.')
print('run status:', run_status)
if hits or run_status != 'finished':
    raise RuntimeError(f'R3 {RUN}: run status {run_status!r}; V10 {"FAILED" if hits else "PASS"}')